In [43]:
#!python -m spacy download en_core_web_sm

In [44]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import nltk
from nltk import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import wordcloud

In [45]:
# # modeli za ustrezno obdelavo stavkov, besed, ločil....
# nltk.download('punkt')     # stavki, besede
# nltk.download('wordnet') #lemmatizacija
# nltk.download('averaged_perceptron_tagger') #POS tagganje
# nltk.download('omw-1.4') 
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

In [46]:
# tokenization and lemmatization 
# lemmatizer= WordNetLemmatizer()

In [47]:
# # pokupčkamo besede s podobnim korenom, pomenom skupaj
# # run, runs, running -> run
# def get_wordnet_pos(treebank_tag):
#     if treebank_tag.startswith('J'):
#         return wordnet.ADJ
#     elif treebank_tag.startswith('V'):
#         return wordnet.VERB
#     elif treebank_tag.startswith('RB'):
#         return wordnet.ADV
#     elif treebank_tag.startswith('N'):
#         return wordnet.NOUN
#     else:
#         return wordnet.NOUN

In [48]:
import spacy

nlp = spacy.load("en_core_web_lg")

In [59]:
def tokenize_lematize(tekst):
    doc = nlp(tekst)
    
    izbrane_besede = []
    
    # želimo odstraniti osebe, kraji, jeziki, narodi...
    odstrani_pos = ['PROPN', 'PRON', 'VERB', 'ADV']
    odstrani_entitete = {'PERSON', 'GPE', 'LOC', 'NORP', 'FAC', 'ORG'}
    
    #mnozica prepoznanih entitet
    for token in doc:
        # odstranimo i
        if token.pos_ in odstrani_pos:
            continue
        if token.ent_type_ in odstrani_entitete:
            continue
            
        # spacy ima oznake NOUN, ADJ
        if token.pos_ in ['NOUN']:
            beseda = token.lemma_.lower()
            # samo crke in dolžina nad 2 znaka
            if beseda.isalpha() and len(beseda) > 2:
                izbrane_besede.append(beseda)
                
    return izbrane_besede

In [60]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()


custom_words = {
    # založniški šum 
    'book', 'novel', 'story', 'author', 'literature', 'edition', 'seller', 
    'read', 'reader', 'page', 'chapter', 'write', 'writer', 'publish', 
    'publication', 'review', 'times', 'york', 'print', 'copy', 'best', 
    'original', 'classic', 'introduction', 'note', 'debut', 'thriller',
    'unique', 'fascinating', 'scandal', 'major', 'character', 'cover', 'magazine',
    'self', 'series', 'volume', 'masterpiece', 'translation', 'film', 'tale', 'course',
    
    # splošni opisi 
    'way', 'thing', 'important', 'practical', 'young', 'boy', 'girl', 
    'human', 'people', 'great', 'good', 'bad', 'true', 'new', 'old',
    'life', 'world', 'everything', 'day', 'time', 'year', 'make',
    'take', 'come', 'think', 'feel', 'know', 'look', 'want', 'large', 'small',
    'man', 'woman', 'literary', 'secret', 'isbn', 'mother', 'sister', 'father',
    'little', 'room', 'place', 'end', 'first', 'second', 'red', 'purple', 'black',
    
    #iz izpisa
    'professional', 'guide', 'experience', 'natural', 'vivid', 'narrative',
    'compelling', 'extraordinary', 'powerful', 'voice', 'mind'
}


all_stopwords = list(base_stopwords.union(custom_words))

In [61]:
# CountVectorizer odstrani 'stopwords' in ustvari nenegativno matriko, na (i, j)-tem mestu
# imamo pojavitev besede i v j-tem dokumentu (glej zapiske na tablici)


# vzamemo 49/50 knjig, eno bomo potem poskusali uvrstiti med žanre
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\najbolj_popularne\naj_ang_opisi\*.txt')[:250]
# min_df=2, max_df=0.9 odstranita redke in pogoste besede, to uniči celoten rezultat
vectorizer= CountVectorizer(stop_words=all_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=3, 
                            max_df=0.75)

In [62]:
X = vectorizer.fit_transform(filepaths) 

c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [63]:
print(X)

# malo lepše, prikaz
dense_matrix = X.toarray()
print(dense_matrix)

#še lepše
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(dense_matrix, columns=feature_names)
print(df.head())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3364 stored elements and shape (250, 572)>
  Coords	Values
  (0, 190)	1
  (0, 124)	3
  (0, 243)	2
  (0, 196)	3
  (0, 289)	1
  (0, 12)	1
  (0, 180)	1
  (0, 497)	2
  (0, 336)	1
  (0, 78)	1
  (0, 241)	1
  (0, 296)	1
  (1, 296)	2
  (1, 571)	1
  (1, 62)	1
  (1, 66)	1
  (1, 224)	1
  (1, 244)	1
  (1, 325)	1
  (2, 336)	1
  (2, 458)	1
  (2, 557)	1
  (2, 392)	1
  (2, 568)	1
  (2, 249)	1
  :	:
  (248, 116)	1
  (248, 173)	2
  (248, 162)	1
  (248, 8)	1
  (248, 202)	2
  (248, 7)	1
  (248, 231)	1
  (248, 384)	1
  (248, 489)	1
  (249, 9)	1
  (249, 283)	2
  (249, 464)	1
  (249, 220)	1
  (249, 22)	1
  (249, 345)	1
  (249, 563)	1
  (249, 565)	1
  (249, 529)	1
  (249, 204)	1
  (249, 564)	1
  (249, 31)	1
  (249, 290)	1
  (249, 52)	1
  (249, 46)	1
  (249, 341)	1
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 0 0 0]
 ...
 [2 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
   academy  accident  account  achievement  act  action  actress  ada

In [64]:
from sklearn.decomposition import PCA

In [72]:
pca = PCA(n_components=4)  
X_pca = pca.fit_transform(X)

In [73]:
print(pca.explained_variance_ratio_)
print("Skupaj:", sum(pca.explained_variance_ratio_))

[0.05109209 0.03493226 0.02195545 0.02053595]
Skupaj: 0.12851574249056663


In [74]:
feature_names = vectorizer.get_feature_names_out()

for i, comp in enumerate(pca.components_):
    top_words = [feature_names[j] for j in comp.argsort()[-10:]]
    print(f"PC{i+1}: {top_words}")

PC1: ['college', 'home', 'love', 'baby', 'money', 'farm', 'wood', 'town', 'family', 'house']
PC2: ['fate', 'war', 'art', 'blood', 'king', 'strigoi', 'vampire', 'friend', 'heart', 'love']
PC3: ['fortune', 'witch', 'popularity', 'power', 'land', 'height', 'love', 'case', 'band', 'adventure']
PC4: ['school', 'magic', 'blood', 'academy', 'princess', 'heart', 'adventure', 'vampire', 'strigoi', 'friend']


In [58]:
# import matplotlib.pyplot as plt
# labels = X_pca.argmax(axis=1)   

# pairs = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]

# for i, j in pairs:
#     plt.figure()
#     plt.scatter(
#         X_pca[:, i],
#         X_pca[:, j],
#         c=labels,       
#         cmap="tab10"    
#     )
#     plt.xlabel(f"PC{i+1}")
#     plt.ylabel(f"PC{j+1}")
#     plt.title(f"PC{i+1} vs PC{j+1}")
#     plt.colorbar(label="žanr")   
#     plt.show()

# plt.show()